# Merge and aggregate all the data

This notebook automates the process of aggregating the CAMELS-GB dataset

In [ ]:
import sys
sys.path.append('../src')

In [ ]:
import os
import requests
import re
from util_IO import get_use_case_main_dir
import pickle
import pandas as pd
from tqdm import tqdm

# Set pandas to display a maximum of 300 columns
pd.set_option('display.max_columns', 300)

## Directories

Derive the directory in order to better manage files locations

In [ ]:
# Directory for use case
camels_gb_use_case_dir = get_use_case_main_dir()

print(f"Main directory for use case:\t\t\t\t{camels_gb_use_case_dir}")


# Directory for dataset
camels_gb_datasets_dir = os.path.join(
    camels_gb_use_case_dir,
    "datasets"
)
print(f"Main directory for datasets:\t\t\t\t{camels_gb_datasets_dir}")


# Main directory for camels_gb - BRONZE LAYER
camels_gb_bronze_dir = os.path.join(
    camels_gb_datasets_dir,
    "camels-gb",
    "data"
)
print(f"Main directory for camels-gb bronze layer:\t\t{camels_gb_bronze_dir}")


# Main directory for camels_gb - SILVER LAYER
camels_gb_silver_dir = os.path.join(
    camels_gb_datasets_dir,
    "camels-gb-aggregated"
)
print(f"Main directory for camels-gb silver layer:\t\t{camels_gb_silver_dir}")


# __________
# Attributes

# Bronze layer 🥉
camels_gb_data_attributes_dir = os.path.join(
    camels_gb_bronze_dir
)
print(f"Directory for camels-gb bronze layer attributes:\t{camels_gb_data_attributes_dir}")


# Silver layer 🥈
camels_gb_data_attributes_aggr_dir = os.path.join(
    camels_gb_silver_dir,
    "attributes"
)
print(f"Directory for camels-gb silver layer attributes:\t{camels_gb_data_attributes_aggr_dir}")


# __________
# Timeseries

# Bronze layer 🥉
camels_gb_data_timeseries_dir = os.path.join(
    camels_gb_bronze_dir,
    "timeseries"
)
print(f"Directory for camels-gb bronze layer timeseries:\t{camels_gb_data_timeseries_dir}")


# Silver layer 🥈
camels_gb_data_timeseries_aggr_dir = os.path.join(
    camels_gb_silver_dir,
    "timeseries"
)
print(f"Directory for camels-gb silver layer timeseries:\t{camels_gb_data_timeseries_aggr_dir}")

## Hydrology Data API

In [ ]:
# Base URL
APIBaseURL = "https://environment.data.gov.uk"

### Functions definitions

In [ ]:
# Retrieve bulk data from REST API
def api_call(url):
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        return f"Failed to retrieve data: {response.status_code}"

# Get nrfaStationID
def get_nrfaStationID(url):
    response = api_call(url)
    if isinstance(response, dict):
        try:
            return response['items'][0]['nrfaStationID']
        except:
            return "No nrfaStationID found"
    else:
        return "Api call failed"


# Define function to get url for query time series
def get_flow_timeseries_url(station):
    return (
        f"{APIBaseURL}/hydrology/id/measures/" +
        f"{station}" + 
        "-flow-m-86400-m3s-qualified/readings.json?" + 
        "maxeq-date=2015-09-30&mineq-date=1970-10-01" # ⚠️ Dates are hardcoded ⚠️
    )

## Files

To manage files programmatically, just the extensions are needed

In [ ]:
# Extensions
attributes_files_ext = ".csv"
timeseries_files_ext = ".csv"

# Regular expression pattern to extract catchmentID
catchmentID_pattern = r"CAMELS_GB_hydromet_timeseries_(.*?)_19701001-20150930"

## Fields management

### Attributes

Common fields definitions/settings

In [ ]:
# Index field
attributes_index = "gauge_id"

Definition of the fields used for each ***level of aggregation***:
 - **full list of fields** (*silver layer*) - no fields selection is made:
   - a pure aggregation with all the fields available from camels-gb dataset
   - `chalk_stream_flag`
 - **fundamental** (*silver layer*) - this level includes:
   - **ONLY** fields which are calculated/derivate from others already present in the camels-gb dataset ( with the exception of `baseflow_index`, for which some variables in camels-gb could be used to calculate it)
   - `chalk_stream_flag`
   - reference for API at `environment.data.gov.uk`

##### `fundamental` level proper attributes (=API excluded)

In [ ]:
# Fields to keep
attributes_fundamental_fields = {
    "hydrologic_attributes_df": [
        'baseflow_index'
    ],
    
    "soil_attributes_df": [
        "sand_perc",
        "silt_perc",
        "clay_perc",
        "organic_perc"
    ],
    
    "climatic_attributes_df": [],
    
    "topographic_attributes_df": [
        "gauge_name",
        "gauge_lat",
        "gauge_lon",
        "gauge_elev",
        "area",
        "dpsbar",
        "elev_mean",
        "elev_min",
        "elev_10",
        "elev_50",
        "elev_90",
        "elev_max"
    ],
    
    "landcover_attributes_df": [
        "dwood_perc",
        "ewood_perc",
        "grass_perc",
        "shrub_perc",
        "crop_perc",
        "urban_perc",
        "inwater_perc",
        "bares_perc"
    ],
    
    "hydrogeology_attributes_df": [],
    
    "humaninfluence_attributes_df": [
        "surfacewater_abs",
        "groundwater_abs",
        "discharges",
        "num_reservoir",
        "reservoir_cap"

    ],
    
    "hydrometry_attributes_df": [
        "bankfull_flow"
    ],

    "chalk_streams_df": [
        "chalk_stream_flag"
    ]
}

### Time series

It is not needed to select any specific field for time series data, as all of the variables are taken in consideration for the analysis, at least during the initial stages. However, it worth to be mentioned that during the process that loads the files, a new field will be added to store the catchment ID; this piece of information is indeed present in the file name only.

In [ ]:
# Date field
date_field = "date"

# Label field
label_field = "discharge_vol"

# Minimum number of samples for group time series 
min_timeseries_length = 5

# Attributes aggregations

## Single files loaded as specific `pd.DataFrame` *(multiple data frames)*

In [ ]:
# Data frame names list initialization (🚩 elements will be strings!)
attributes_df_names_list = []

# Loop through all files in the directory for camels-gb attributes
for filename in os.listdir(camels_gb_data_attributes_dir):
    
    # Define current full path
    path = os.path.join(camels_gb_data_attributes_dir, filename)
    
    # Check if it's a file and has the desired extension
    if os.path.isfile(path) and filename.endswith(attributes_files_ext):
    
        # Define current table name via file name extraction
        curr_df_name = f"{filename[10:-4]}_df"
        print(curr_df_name)
        attributes_df_names_list.append(curr_df_name)

        # Read the file into a DataFrame
        df = pd.read_csv(
            path,
            dtype={attributes_index: 'str'},
            index_col=attributes_index
        )
    
        # Dynamically create a variable with `curr_df_name` as name
        globals()[curr_df_name] = df
        display(df.head(3))

## File for chalk streams

In [ ]:
# Define table name
path = (
    os.path.join(
        camels_gb_datasets_dir,
        "chalk_streams.csv"
    )
)

# Add to attributes data frame list (of names!)
attributes_df_names_list.append("chalk_streams_df")

# Read the file into a DataFrame
chalk_streams_df = pd.read_csv(
    path,
    dtype={attributes_index: 'str'},
    index_col=attributes_index
)

print(attributes_df_names_list[-1])
display(chalk_streams_df.head(3))

## Regroup variables

In [ ]:
# List the data frame (🚩 elements will be data frames!)
attributes_df_list = [globals()[name] for name in attributes_df_names_list]

# Length of the list of data frame
n_df = len(attributes_df_list)

## Check on indices

In [ ]:
# Use the first data frame to infer the list of item
reference_index = attributes_df_list[0].index

for i, df in enumerate(attributes_df_list):
    assert df.index.equals(reference_index), f"Data frame {attributes_df_names_list[i]} has a different index."

## `full_list_of_fields` aggregation

In [ ]:
# Initialization of the aggregation data frame
full_list_of_fields_df = attributes_df_list[0]

# Loop on joining data frames
for i in range(1,n_df):
    
    # Join current data frame with the following
    full_list_of_fields_df = (
        full_list_of_fields_df.
            join(attributes_df_list[i])
    )

### Checks on `full_list_of_fields` aggregation

In [ ]:
assert len(full_list_of_fields_df.columns) == sum(len(df.columns) for df in attributes_df_list), (
    "Total number of columns for aggregated data frame does NOT match the sum of columns numerosity of the original data frames"
)

In [ ]:
display(full_list_of_fields_df.head(3))

In [ ]:
print(f"Number of columns for `full_list_of_fields` data frame: {len(full_list_of_fields_df.columns)}")

### Save `full_list_of_fields`

In [ ]:
# Define path to save
path = os.path.join(
        camels_gb_data_attributes_aggr_dir,
        "full_list_of_fields.csv"
)

# Save
full_list_of_fields_df.to_csv(path)

## `fundamental` aggregation

In [ ]:
# First set of columns
curr_columns_set = attributes_fundamental_fields[attributes_df_names_list[0]]

# Initialization of the aggregation data frame
fundamental_df = attributes_df_list[0][curr_columns_set]

# Loop on joining data frames
for i in range(1,n_df):
    
    # Join current data frame with the following
    fundamental_df = (
        fundamental_df.
            join(
                attributes_df_list[i][
                    attributes_fundamental_fields[attributes_df_names_list[i]]
                ]
            )
    )

### Checks on `fundamental` aggregation

In [ ]:
assert len(fundamental_df.columns) == sum(len(fields_list) for fields_list in attributes_fundamental_fields.values()), (
    "Total number of columns for aggregated data frame does NOT match the sum of columns inside the dictionary stating fundamental attributes fields"
)

In [ ]:
display(fundamental_df.head(3))

In [ ]:
print(f"Number of columns for `fundamental` data frame: {len(fundamental_df.columns)}")

### API catchments set

#### Retrieve full list of `nrfaStationID` codes

In [ ]:
# _______________________________________
# Retrieve the list of available stations
url = f"{APIBaseURL}/hydrology/id/stations.json?_limit=100000"

# Call to review ALL the stations
json_data = api_call(url)
print(f"N. of stations found:\t{len(json_data['items'])}")

# ___________________________________________
# Set dataFrames to collect station meta date
full_stations_OpenAPI_df = pd.DataFrame({
    '@id': [item['@id'] for item in json_data['items']],
    'label': [item['label'] for item in json_data['items']]
})

# Create api for specific call on station details
full_stations_OpenAPI_df['query_for_station_specifics'] = full_stations_OpenAPI_df['@id'] + ".json"

# ________________________
# Retrieve `nrfaStationID` with progress bar
tqdm.pandas(desc="Retrieving nrfaStationID")
full_stations_OpenAPI_df['nrfaStationID'] = full_stations_OpenAPI_df['query_for_station_specifics'].progress_apply(get_nrfaStationID)

# Quick overview
print(full_stations_OpenAPI_df['nrfaStationID'].value_counts())

#### Manage problematic cases

In [ ]:
# Display duplicates
print(
    full_stations_OpenAPI_df
    .loc[lambda df: df['nrfaStationID'] != 'No nrfaStationID found']
    .loc[lambda df: df['nrfaStationID'].isin(df['nrfaStationID'].value_counts()[lambda x: x != 1].index)]
    .sort_values(by='nrfaStationID')
    .values
)

In [ ]:
# Dropping problematic `nrfaStationID``
unique_stations_OpenAPI_df = (
    full_stations_OpenAPI_df
        .loc[lambda df: df['nrfaStationID'] != 'No nrfaStationID found']
        .loc[lambda df: df['nrfaStationID'].isin(df['nrfaStationID'].value_counts()[lambda x: x == 1].index)]
)

#### Join API and CAMELS_GB ***stations***

In [ ]:
# INNER JOIN with attributes data frame
print(f"N. of catchments in attributes table BEFORE the merge:\t{fundamental_df.shape[0]}")

fundamental_api_df = (
    fundamental_df
        .merge(
            unique_stations_OpenAPI_df,
            how='inner',
            left_index=True,
            right_on='nrfaStationID'
        )
        .set_index('nrfaStationID')
        .rename_axis(attributes_index)
)

print(f"N. of catchments remained AFTER the merge:\t\t{fundamental_api_df.shape[0]}")

#### Derive URL for time series (`measures`)

In [ ]:
# Derive station code
fundamental_api_df['station_code'] = fundamental_api_df['@id'].str.split('/').str[-1]

# Derive URL for time series query
fundamental_api_df['query_for_station_specifics'] = fundamental_api_df['station_code'].apply(get_flow_timeseries_url)

display(fundamental_api_df.head())

### Save `fundamental`

In [ ]:
# Define path to save
path = os.path.join(
        camels_gb_data_attributes_aggr_dir,
        "fundamental.csv"
)

# Save
fundamental_api_df.to_csv(path)

# Timeseries aggregations

## `timeseries` aggregation

In [ ]:
# Create an empty data frame to store timeseries
timeseries_df = pd.DataFrame()

# Create an empty dictionary to store potential aggregation issues 
timeseries_aggr_issues_dict = {}

# Create counter for unmatched catchments
i = 1

print(f"Processing catchments: ", end="")

# Loop through all files in the directory for camels-gb time series
for filename in os.listdir(camels_gb_data_timeseries_dir):
    
    # Define current full path
    path = os.path.join(camels_gb_data_timeseries_dir, filename)
    
    # Check if it's a file and has the desired extension
    if os.path.isfile(path) and filename.endswith(timeseries_files_ext):
    
        # Extract catchmentID
        match = re.search(catchmentID_pattern, filename)
        
        # When a match is found..
        if match:
            catchmentID = match.group(1)
            print(f"ID: {catchmentID} ...   ", end="")
        
            # Read the file into a DataFrame
            df = pd.read_csv(
                path,
                parse_dates=[date_field]
            )
            
            # Convert from Timestamp to datetime.date
            df[date_field] = df[date_field].dt.date

            # Add catchment ID (as first column)
            df.insert(0, 'catchmentID', catchmentID)
            
            # Aggregation
            timeseries_df = pd.concat([timeseries_df, df], ignore_index=True)
            
            print("Ok!  ", end="")

        #..otherwise notify which is the problematic file..
        #..and skip to the following
        else:
            print(f"Catchment ID not found for file {filename}  ", end="")
            timeseries_aggr_issues_dict[f"Unknown_{i}"] = {
                    "File": path,
                    "Issue": "File skipped because not able to retrieve catchmentID from file name"
            }
            i += 1
            continue

            
# Final order by `catchmentID`
print("\nFinal sorting..")
timeseries_df.sort_values(
    by=["catchmentID", date_field],
    inplace=True,
    ignore_index=True
)

print("Aggregation complete!")

# Display issues, if any
if timeseries_aggr_issues_dict:
    print(f"Please have a look at the following issues during the aggregation process:\n{timeseries_aggr_issues_dict}")

### Checks on `timeseries_df` aggregation

In [ ]:
display(timeseries_df)

### Download API time series with `Good` and `Complete` quality flags

#### Download

In [ ]:
# Flow timeseries API data frame
discharge_api_df = pd.DataFrame()

# Loop through the DataFrame
for index, row in fundamental_api_df.iterrows():

    # Create a DataFrame with the data from the API call
    curr_df = pd.DataFrame(
        api_call(
            row['query_for_station_specifics']
        )
        ['items']
    )

    # Add the station code
    curr_df.insert(0, 'catchmentID', index)

    # Concatenate with the main data frame
    discharge_api_df = pd.concat(
        [
            discharge_api_df,
            curr_df
        ],
        axis=0
    )

# Make-up
discharge_api_df.rename(columns={'value': label_field}, inplace=True)
discharge_api_df['date'] = pd.to_datetime(discharge_api_df['date']).dt.date

#### `Good` and `Complete` only

In [ ]:
discharge_api_df = (
    discharge_api_df[
        (discharge_api_df['completeness'] == 'Complete') &
            (discharge_api_df['quality'] == 'Good')
    ]
)

display(discharge_api_df.head())

In [ ]:
# Numerosity BEFORE merge
print(f"CAMELS-GB:\t{timeseries_df.shape[0]:,}")
print(f"API:\t\t{discharge_api_df.shape[0]:,}")

### Join API with CAMELS_GB ***timeseries***

In [ ]:
# `discharge_vol` values check
timeseries_api_df = (
    timeseries_df
    .merge(
        discharge_api_df[['catchmentID', 'date', label_field]],
        how='left',
        left_on=['catchmentID', date_field],
        right_on=['catchmentID', date_field],
        suffixes=('_files', '_api')
    )
)

# Drop NaNs
timeseries_api_df.dropna(inplace=True)

# Make-up
timeseries_api_df.rename(
    columns={
        f"{label_field}_api": label_field
    },
    inplace=True
)

timeseries_api_df.reset_index(
    inplace=True,
    drop=True
)
display(timeseries_api_df.head())

In [ ]:
# N. catchments after API's quality checks
print(f"N. of catchments with API's quality checks:\t{timeseries_api_df['catchmentID'].nunique()}")

### Split of catchments for not continuous time series

In [ ]:
# Create en empty list to store catchment groups with too short time series
catchmentsID_ts_too_short_list = []

def split_timeseries(group):

    # Retrieving current catchment ID
    curr_catchmentID = group['catchmentID'].iloc[0]

    # Calculate the difference in days
    group[f"{date_field}_diff"] = group[date_field].diff().dt.days

    # Fill NaN values with 0
    group[f"{date_field}_diff"] = group[f"{date_field}_diff"].fillna(0)

    # Determine if the date is the same as the previous day
    group[f"{date_field}_consecutive_day"] = group[f"{date_field}_diff"] > 1

    # Calculate the cumulative sum to create groups
    group[f"{date_field}_group"] = group[f"{date_field}_consecutive_day"].cumsum()

    # Convert group into string
    group[f"{date_field}_group"] = (
        group[f"{date_field}_group"]
            .astype(str)
            .str
            .zfill(2)
    )

    # Calculate the size of each group
    group_sizes = group[f"{date_field}_group"].value_counts()

    # Save list of catchments with too short time series
    (
        catchmentsID_ts_too_short_list.extend([
            f"{curr_catchmentID}-{x}" for x in
                group_sizes[
                    group_sizes < min_timeseries_length
                ]
                .index
                .to_list()
        ])
    )

    # Remove `group` with less than `min_timeseries_length`
    filtered_group  = (
        group[
            group[f"{date_field}_group"]
                .isin(
                    group_sizes[
                        group_sizes >= min_timeseries_length
                    ].index
                )
        ]
    )

    return filtered_group

# Apply the function to each sensor
timeseries_api_df = (
    timeseries_api_df
        .groupby('catchmentID')
        .apply(split_timeseries)
        .reset_index(drop=True)
)

display(timeseries_api_df.head())

### Save `timeseries_df`

In [ ]:
# Define path to save
path = os.path.join(
        camels_gb_data_timeseries_aggr_dir,
        "timeseries.csv"
)

# Save
timeseries_api_df.to_csv(
    path,
    index=False
)

# Metadata

## Store parameters used for aggregations

In [ ]:
# Create dictionary
aggr_parameters_dict = {
    "camels_gb_use_case_dir": camels_gb_use_case_dir,
    "camels_gb_datasets_dir": camels_gb_datasets_dir,
    "camels_gb_bronze_dir": camels_gb_bronze_dir,
    "camels_gb_silver_dir": camels_gb_silver_dir,
    "camels_gb_data_attributes_dir": camels_gb_data_attributes_dir,
    "camels_gb_data_attributes_aggr_dir": camels_gb_data_attributes_aggr_dir,
    "camels_gb_data_timeseries_dir": camels_gb_data_timeseries_dir,
    "camels_gb_data_timeseries_aggr_dir": camels_gb_data_timeseries_aggr_dir,
    "attributes": {
        "attributes_files_ext": attributes_files_ext,
        "attributes_index": attributes_index,
        "aggregations": {
            "fundamental": attributes_fundamental_fields
        }
    },
    "timeseries": {
        "timeseries_files_ext": timeseries_files_ext,
        "catchmentID_pattern": catchmentID_pattern,
        "date_field": date_field,
        "label_field": label_field,
        "timeseries_aggr_issues_dict": timeseries_aggr_issues_dict
    }
}

# Store dictionary
with open(
    os.path.join(
        camels_gb_use_case_dir,
        'aggr_parameters_dict.pkl'
    ),
    'wb'
) as f:
    pickle.dump(aggr_parameters_dict, f)

## Store dictionary with time range per `catchment`-`group` time series

In [ ]:
# Derive statistics about length (as data frame)
length_df = (
    timeseries_api_df
        .groupby(
            ['catchmentID', f"{date_field}_group"]
        )
        .agg(
            start_date=(date_field, 'min'),
            end_date=(date_field, 'max'),
            length=(date_field, 'count')
        )
        .reset_index()
)


# Create a dictionary properly formatted
length_dict = length_df.set_index(['catchmentID', f"{date_field}_group"]).to_dict(orient='index')
catchmentID_time_ranges_dict = {
    f"{key[0]}-{key[1]}": value
    for key, value in length_dict.items()
}


# Store
with open(
    os.path.join(
        camels_gb_use_case_dir,
        'catchmentID_time_ranges_dict.pkl'
    ),
    'wb'
) as f:
    pickle.dump(catchmentID_time_ranges_dict, f)

## Store list for too short `catchment`-`group` time series

In [ ]:
with open(
    os.path.join(
        camels_gb_use_case_dir,
        f"catchmentsID_ts_shorter_than_{min_timeseries_length}.pkl"
    ),
    'wb'
) as f:
    pickle.dump(catchmentsID_ts_too_short_list, f)